# Module 4 — Forward Kinematics using Denavit–Hartenberg Parameters
## Unit 6 — Building and Using a DH Table
### Lesson 6.3 — DH Forward Kinematics in Code

*Physical AI Curriculum · hands-on notebook. Run **Kernel → Restart & Run All**.*

In [ ]:
import numpy as np
from functools import reduce
def rotz(t):
    c,s=np.cos(t),np.sin(t); return np.array([[c,-s,0,0],[s,c,0,0],[0,0,1,0],[0,0,0,1.0]])
def transz(d):
    T=np.eye(4); T[2,3]=d; return T
def transx(a):
    T=np.eye(4); T[0,3]=a; return T
def rotx(al):
    c,s=np.cos(al),np.sin(al); return np.array([[1,0,0,0],[0,c,-s,0],[0,s,c,0],[0,0,0,1.0]])
def dh_transform(theta,d,a,alpha):
    return rotz(theta)@transz(d)@transx(a)@rotx(alpha)
def dh_fk(table,config):
    # table rows: (d,a,alpha,kind); fill variable, build, multiply
    factors=[]
    for (d,a,alpha,kind),q in zip(table,config):
        theta = q if kind=='revolute' else 0.0
        dd = q if kind=='prismatic' else d
        factors.append(dh_transform(theta,dd,a,alpha))
    T=reduce(lambda A,B:A@B, factors, np.eye(4))
    return T
arm3=[(0.1,0,np.radians(90),'revolute'),(0,0.4,0,'revolute'),(0,0.3,0,'revolute')]
T=dh_fk(arm3,[np.radians(0),np.radians(30),np.radians(60)])
print('gripper position',np.round(T[:3,3],3))

## Coding Exercise (§8)
Verify the 3-DOF in-plane reach matches the planar 2-link formula; then derive symbolic DH FK with SymPy.

In [ ]:
# YOUR CODE HERE
import numpy as np
# With base swivel theta1=0 and alpha1=90, the two planar links reach in a vertical plane.
# The in-plane reach distance must equal the planar 2-link reach for (30,60).
def planar_reach(t1,t2,L1,L2):
    x=L1*np.cos(t1)+L2*np.cos(t1+t2); y=L1*np.sin(t1)+L2*np.sin(t1+t2); return np.hypot(x,y)
p=T[:3,3]; horiz=np.hypot(p[0],p[1]); 
reach_planar=planar_reach(np.radians(30),np.radians(60),0.4,0.3)
# the planar reach (0.346,0.5) has magnitude ~0.609; check the arm's distance from the z0 axis matches
assert np.isclose(np.hypot(horiz, p[2]-0.1), reach_planar, atol=1e-6)
import sympy as sp
th1,th2,th3=sp.symbols('theta1 theta2 theta3',real=True)
def srotz(t): return sp.Matrix([[sp.cos(t),-sp.sin(t),0,0],[sp.sin(t),sp.cos(t),0,0],[0,0,1,0],[0,0,0,1]])
def stz(d): return sp.Matrix([[1,0,0,0],[0,1,0,0],[0,0,1,d],[0,0,0,1]])
def stx(a): return sp.Matrix([[1,0,0,a],[0,1,0,0],[0,0,1,0],[0,0,0,1]])
def srotx(al): return sp.Matrix([[1,0,0,0],[0,sp.cos(al),-sp.sin(al),0],[0,sp.sin(al),sp.cos(al),0],[0,0,0,1]])
def sdh(t,d,a,al): return srotz(t)*stz(d)*stx(a)*srotx(al)
Ts=sdh(th1,sp.Rational(1,10),0,sp.pi/2)*sdh(th2,0,sp.Rational(2,5),0)*sdh(th3,0,sp.Rational(3,10),0)
Tnum=np.array(Ts.subs({th1:0,th2:sp.rad(30),th3:sp.rad(60)}).evalf(),float)
assert np.allclose(Tnum[:3,3],T[:3,3],atol=1e-6)
print('DH FK in-plane reach matches planar formula; symbolic DH agrees numerically.')

## Check your work

In [ ]:
import numpy as np
assert T.shape==(4,4) and np.isclose(T[3,3],1.0)
print('All checks passed.')